# Figures S3–S4

In [ ]:
from pathlib import Path
import json
import re
import time

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy import sparse
from scipy.interpolate import griddata
from matplotlib.ticker import MaxNLocator

from discretize import TensorMesh
from discretize.utils import active_from_xyz
from simpeg import maps, regularization
from simpeg.potential_fields import gravity, magnetics


# This notebook recomputes the sensitivity weighting used by the existing
# joint inversions.  It does not call an optimizer or modify inversion results.
ROOT = Path.cwd()
FIGURE_DIR = ROOT / "Figure"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CASES = {
    "Hannah": {
        "directory": ROOT / "Hannah_Inversion_GPT",
        "y_targets_m": (4293250.0, 4303250.0, 4313250.0),
        "regularized_vmin": -6.0,
        "target_intervals_km": {"shallow": (1.0, 4.0), "deep": (4.0, 7.0)},
        "output": FIGURE_DIR / "FigureS3_senstivity_hannah.png",
    },
    "Iowa": {
        "directory": ROOT / "Iowa_Inversion_GPT",
        "y_targets_m": (4792250.0, 4804250.0, 4819250.0),
        "regularized_vmin": -1.0,
        "target_intervals_km": {"shallow": (0.0, 3.0), "mid": (3.0, 7.0), "deep": (7.0, np.inf)},
        "output": FIGURE_DIR / "FigureS4_senstivity_Iowa.png",
    },
}

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 15.5,
    "axes.unicode_minus": False,
    "axes.linewidth": 0.8,
    "xtick.direction": "out",
    "ytick.direction": "out",
})


def nearest_indices(values, targets):
    return [int(np.argmin(np.abs(values - target))) for target in targets]


def core_indices(full_mesh, core_mesh, axis):
    full_values = getattr(full_mesh, f"cell_centers_{axis}")
    core_values = getattr(core_mesh, f"cell_centers_{axis}")
    indices = np.array([int(np.argmin(np.abs(full_values - value))) for value in core_values])
    if not np.allclose(full_values[indices], core_values, atol=1e-6):
        raise ValueError(f"Core {axis}-coordinates do not align with the full inversion mesh.")
    return indices


def active_vector_to_core(full_mesh, core_mesh, active_cells, active_values):
    if active_values.size != int(active_cells.sum()):
        raise ValueError("Sensitivity vector length does not match the active-cell mask.")
    full_values = np.full(full_mesh.nC, np.nan, dtype=float)
    full_values[active_cells] = active_values
    full_3d = full_values.reshape(full_mesh.shape_cells, order="F")
    ix = core_indices(full_mesh, core_mesh, "x")
    iy = core_indices(full_mesh, core_mesh, "y")
    iz = core_indices(full_mesh, core_mesh, "z")
    return full_3d[np.ix_(ix, iy, iz)]


def final_betas(project_dir):
    """Read the last paired beta values recorded by the archived inversion."""
    output_files = sorted((project_dir / "iteration_model").glob("Output_*.txt"))
    if not output_files:
        raise FileNotFoundError(f"No archived iteration output found in {project_dir}.")
    lines = [line for line in output_files[-1].read_text(encoding="utf-8").splitlines() if line.strip() and not line.lstrip().startswith("#")]
    match = re.search(r"\[\s*'([^']+)'\s*,\s*'([^']+)'\s*\]", lines[-1])
    if match is None:
        raise ValueError(f"Could not read paired betas from {output_files[-1]}.")
    return np.array([float(match.group(1)), float(match.group(2))]), output_files[-1]


def sparse_regularization_diagonal(reg, model):
    """Diagonal of the quadratic curvature from one archived Sparse regularizer."""
    diagonal = np.zeros(model.size, dtype=float)
    for multiplier, term in zip(reg.multipliers, reg.objfcts):
        diagonal += multiplier * term.deriv2(model).diagonal()
    return diagonal


def normalize_for_display(values):
    peak = np.nanmax(values)
    if not np.isfinite(peak) or peak <= 0:
        raise ValueError("Sensitivity/curvature has no positive finite values.")
    return values / peak


def depth_below_topography_km(core_mesh, topography):
    """Return the depth below the interpolated local topographic surface for every core cell."""
    core_x, core_y = np.meshgrid(core_mesh.cell_centers_x, core_mesh.cell_centers_y, indexing="ij")
    surface = griddata(topography[:, :2], topography[:, 2], (core_x, core_y), method="linear")
    if np.isnan(surface).any():
        nearest = griddata(topography[:, :2], topography[:, 2], (core_x, core_y), method="nearest")
        surface = np.where(np.isnan(surface), nearest, surface)
    return (surface[:, :, None] - core_mesh.cell_centers_z[None, None, :]) / 1000.0


def depth_thresholds_km(sensitivity, depth_km, bin_width_km=0.25):
    """Depth where the laterally median normalized sensitivity remains below each threshold."""
    valid = np.isfinite(sensitivity) & np.isfinite(depth_km) & (depth_km >= 0.0)
    max_depth = float(np.nanmax(depth_km[valid]))
    edges = np.arange(0.0, max_depth + bin_width_km, bin_width_km)
    centers = 0.5 * (edges[:-1] + edges[1:])
    medians = np.full(centers.size, np.nan)
    for index, (lower, upper) in enumerate(zip(edges[:-1], edges[1:])):
        in_bin = valid & (depth_km >= lower) & (depth_km < upper)
        if np.any(in_bin):
            medians[index] = np.nanmedian(sensitivity[in_bin])
    finite = np.isfinite(medians) & (medians > 0.0)
    thresholds = {}
    for threshold in (1e-1, 1e-2, 1e-3):
        below = finite & (medians <= threshold)
        stable_below = np.logical_and.accumulate(below[::-1])[::-1]
        candidates = np.flatnonzero(stable_below)
        if candidates.size == 0:
            thresholds[threshold] = np.nan
            continue
        index = int(candidates[0])
        if index == 0 or not finite[index - 1]:
            thresholds[threshold] = centers[index]
            continue
        previous_log = np.log10(medians[index - 1])
        current_log = np.log10(medians[index])
        target_log = np.log10(threshold)
        fraction = 0.0 if np.isclose(previous_log, current_log) else (target_log - previous_log) / (current_log - previous_log)
        thresholds[threshold] = centers[index - 1] + np.clip(fraction, 0.0, 1.0) * (centers[index] - centers[index - 1])
    return thresholds


def interval_sensitivity_statistics(sensitivity, depth_km, lower_km, upper_km):
    valid = np.isfinite(sensitivity) & np.isfinite(depth_km) & (depth_km >= lower_km) & (depth_km < upper_km)
    values = np.log10(sensitivity[valid])
    return {"median": float(np.nanmedian(values)), "p10": float(np.nanpercentile(values, 10)), "p90": float(np.nanpercentile(values, 90)), "n_cells": int(values.size)}


def interval_data_fraction_percent(data_fraction, depth_km, lower_km, upper_km):
    valid = np.isfinite(data_fraction) & np.isfinite(depth_km) & (depth_km >= lower_km) & (depth_km < upper_km)
    values = 100.0 * data_fraction[valid]
    return {"median_percent": float(np.nanmedian(values)), "n_cells": int(values.size)}


def sensitivity_summary(case_name, spec, result):
    """Statistics used in the accompanying results text; all depths are below local topography."""
    depth_km = depth_below_topography_km(result["core_mesh"], result["topography"])
    summary = {
        "case": case_name,
        "depth_reference": "below interpolated local topography",
        "thresholds_km": {
            "gravity": depth_thresholds_km(result["core_maps"]["gravity"], depth_km),
            "magnetic": depth_thresholds_km(result["core_maps"]["magnetic"], depth_km),
        },
        "intervals": {},
    }
    for label, (lower_km, upper_km) in spec["target_intervals_km"].items():
        summary["intervals"][label] = {
            "depth_range_km": (lower_km, upper_km),
            "gravity_log10": interval_sensitivity_statistics(result["core_maps"]["gravity"], depth_km, lower_km, upper_km),
            "magnetic_log10": interval_sensitivity_statistics(result["core_maps"]["magnetic"], depth_km, lower_km, upper_km),
            "data_fraction": interval_data_fraction_percent(result["core_data_fraction"], depth_km, lower_km, upper_km),
        }
    return summary


def compute_joint_sensitivity(case_name, spec):
    """Rebuild the saved forward operators and calculate their RMS sensitivity weights."""
    project_dir = spec["directory"]
    params = json.loads((project_dir / "inversion_params.json").read_text(encoding="utf-8"))
    full_mesh = TensorMesh.read_UBC(project_dir / "mesh" / "mesh.msh")
    core_mesh = TensorMesh.read_UBC(project_dir / "mesh" / "mesh_core.msh")
    topography = np.loadtxt(project_dir / "topo" / "topography.xyz", ndmin=2)
    gravity_obs = np.loadtxt(project_dir / "observed_data" / "gravity.obs", ndmin=2)
    magnetic_obs = np.loadtxt(project_dir / "observed_data" / "magnetics.obs", ndmin=2)

    if gravity_obs.shape[1] < 5 or magnetic_obs.shape[1] < 5:
        raise ValueError(f"{case_name}: saved observations must contain x, y, z, data, and standard deviation.")

    active_cells = active_from_xyz(full_mesh, topography)
    n_active = int(active_cells.sum())
    wires = maps.Wires(("density", n_active), ("susceptibility", n_active))
    density_model = TensorMesh.read_model_UBC(
        full_mesh, project_dir / "inversion_result" / "joint_density_full_UBC.txt"
    )[active_cells]
    susceptibility_model = TensorMesh.read_model_UBC(
        full_mesh, project_dir / "inversion_result" / "joint_susceptibility_full_UBC.txt"
    )[active_cells]
    model = np.r_[density_model, susceptibility_model]

    gravity_receivers = gravity.receivers.Point(
        gravity_obs[:, :3], components=params["gravity_component"]
    )
    gravity_survey = gravity.survey.Survey(
        gravity.sources.SourceField(receiver_list=[gravity_receivers])
    )
    magnetic_receivers = magnetics.receivers.Point(magnetic_obs[:, :3], components="tmi")
    magnetic_survey = magnetics.survey.Survey(
        magnetics.sources.UniformBackgroundField(
            receiver_list=[magnetic_receivers],
            amplitude=params["field_strength"],
            inclination=params["inclination"],
            declination=params["declination"],
        )
    )

    gravity_simulation = gravity.simulation.Simulation3DIntegral(
        survey=gravity_survey,
        mesh=full_mesh,
        rhoMap=wires.density,
        active_cells=active_cells,
        engine="choclo",
    )
    magnetic_simulation = magnetics.simulation.Simulation3DIntegral(
        survey=magnetic_survey,
        mesh=full_mesh,
        model_type="scalar",
        chiMap=wires.susceptibility,
        active_cells=active_cells,
        engine="choclo",
    )

    start = time.perf_counter()
    gravity_jtj = gravity_simulation.getJtJdiag(
        model, W=sparse.diags(1.0 / gravity_obs[:, 4])
    )
    magnetic_jtj = magnetic_simulation.getJtJdiag(
        model, W=sparse.diags(1.0 / magnetic_obs[:, 4])
    )
    elapsed_s = time.perf_counter() - start

    # Rows 1 and 2 use the RMS/volume sensitivity weights used by
    # UpdateSensitivityWeights for the gravity and magnetic data terms.
    volumes = full_mesh.cell_volumes[active_cells]
    gravity_rms = np.sqrt(np.clip(gravity_jtj[:n_active], 0.0, None)) / volumes
    magnetic_rms = np.sqrt(np.clip(magnetic_jtj[n_active:], 0.0, None)) / volumes

    # Rebuild the archived regularization operators at the saved final model.
    # The third row is the diagonal local curvature of the actual objective:
    # data terms + final paired-beta Sparse terms + lambda CrossGradient term.
    reg_gravity = regularization.Sparse(
        full_mesh, active_cells=active_cells, mapping=wires.density, gradient_type="components"
    )
    reg_magnetic = regularization.Sparse(
        full_mesh, active_cells=active_cells, mapping=wires.susceptibility, gradient_type="components"
    )
    for reg, norms, alphas in (
        (reg_gravity, params["reg_grv_norm"], params["weight_grv"]),
        (reg_magnetic, params["reg_mag_norm"], params["weight_mag"]),
    ):
        reg.norms = norms
        reg.alpha_s, reg.alpha_x, reg.alpha_y, reg.alpha_z = alphas

    betas, beta_output = final_betas(project_dir)
    gravity_reg_diag = sparse_regularization_diagonal(reg_gravity, model)[:n_active]
    magnetic_reg_diag = sparse_regularization_diagonal(reg_magnetic, model)[n_active:]
    cross_gradient = regularization.CrossGradient(full_mesh, wires, active_cells=active_cells)
    cross_gradient_diag = cross_gradient.deriv2(model).diagonal()
    lambda_cross_gradient = float(params["cross_gradient_lambda"])

    density_curvature = gravity_jtj[:n_active] + betas[0] * gravity_reg_diag + lambda_cross_gradient * cross_gradient_diag[:n_active]
    susceptibility_curvature = magnetic_jtj[n_active:] + betas[1] * magnetic_reg_diag + lambda_cross_gradient * cross_gradient_diag[n_active:]
    data_diagonal = gravity_jtj[:n_active] + magnetic_jtj[n_active:]
    total_diagonal = density_curvature + susceptibility_curvature
    data_fraction = np.divide(data_diagonal, total_diagonal, out=np.full(n_active, np.nan), where=total_diagonal > 0.0)
    joint_regularized_rms = np.sqrt(
        np.clip(density_curvature, 0.0, None) / volumes**2
        + np.clip(susceptibility_curvature, 0.0, None) / volumes**2
    )

    core_maps = {
        "gravity": active_vector_to_core(full_mesh, core_mesh, active_cells, normalize_for_display(gravity_rms)),
        "magnetic": active_vector_to_core(full_mesh, core_mesh, active_cells, normalize_for_display(magnetic_rms)),
        "joint_regularized": active_vector_to_core(full_mesh, core_mesh, active_cells, normalize_for_display(joint_regularized_rms)),
    }

    return {
        "core_mesh": core_mesh,
        "core_maps": core_maps,
        "core_data_fraction": active_vector_to_core(full_mesh, core_mesh, active_cells, data_fraction),
        "topography": topography,
        "betas": betas,
        "beta_output": beta_output,
        "lambda_cross_gradient": lambda_cross_gradient,
        "elapsed_s": elapsed_s,
        "n_active": n_active,
        "n_gravity": gravity_obs.shape[0],
        "n_magnetic": magnetic_obs.shape[0],
        "gravity_component": params["gravity_component"],
    }


def plot_sensitivity(case_name, spec, result):
    mesh = result["core_mesh"]
    x_edges_km = mesh.nodes_x / 1000.0
    z_edges_km = mesh.nodes_z / 1000.0
    j_indices = nearest_indices(mesh.cell_centers_y, spec["y_targets_m"])

    row_specs = (
        ("Gravity data\nsensitivity", result["core_maps"]["gravity"]),
        ("Magnetic data\nsensitivity", result["core_maps"]["magnetic"]),
        ("Joint +\nregularization\ncurvature", result["core_maps"]["joint_regularized"]),
    )
    cmap = mpl.colormaps["magma"].copy()
    cmap.set_bad("white")

    fig = plt.figure(figsize=(22.0, 14.0))
    grid = fig.add_gridspec(
        3, 4, width_ratios=(1.0, 1.0, 1.0, 0.05),
        left=0.13, right=0.95, bottom=0.075, top=0.88, wspace=0.13, hspace=0.32,
    )
    axes = np.empty((3, 3), dtype=object)
    for row in range(3):
        for column in range(3):
            axes[row, column] = fig.add_subplot(grid[row, column])
    data_colorbar_axis = fig.add_subplot(grid[:2, 3])
    regularized_colorbar_axis = fig.add_subplot(grid[2, 3])
    letters = iter("abcdefghi")
    images = []

    for column, j in enumerate(j_indices):
        position = axes[0, column].get_position()
        fig.text(
            (position.x0 + position.x1) / 2, position.y1 + 0.015,
            f"Northing = {mesh.cell_centers_y[j] / 1000.0:.3f} km",
            ha="center", va="bottom", fontsize=17, fontweight="bold",
        )

    for row, (row_label, sensitivity) in enumerate(row_specs):
        row_vmin = -6.0 if row < 2 else spec["regularized_vmin"]
        display = np.log10(np.clip(sensitivity, 1e-6, None))
        for column, j in enumerate(j_indices):
            axis = axes[row, column]
            image = axis.pcolormesh(
                x_edges_km, z_edges_km, display[:, j, :].T,
                shading="flat", cmap=cmap, vmin=row_vmin, vmax=0.0, rasterized=True,
            )
            axis.text(
                0.0, 1.02, f"({next(letters)})", transform=axis.transAxes,
                ha="left", va="bottom", fontsize=17, fontweight="bold", clip_on=False,
            )
            axis.set_xlabel("Easting (km)", fontsize=16)
            if column == 0:
                axis.set_ylabel("Elevation (km)", fontsize=16)
            else:
                axis.tick_params(labelleft=False)

        images.append(image)
        row_center = np.mean([axes[row, 0].get_position().y0, axes[row, 0].get_position().y1])
        fig.text(0.055, row_center, row_label, va="center", ha="center", fontsize=16, fontweight="bold")

    for axes_row in axes:
        for axis in axes_row:
            axis.xaxis.set_major_locator(MaxNLocator(nbins=4))
            axis.yaxis.set_major_locator(MaxNLocator(nbins=5))
            axis.tick_params(labelsize=13.5, width=0.9, length=4)
            for spine in axis.spines.values():
                spine.set_visible(True)
            axis.set_aspect("auto")

    data_colorbar = fig.colorbar(images[1], cax=data_colorbar_axis)
    data_colorbar.set_label(r"log10(relative data sensitivity)", fontsize=15.5)
    data_colorbar.set_ticks(np.arange(-6, 1, 1))
    data_colorbar.ax.tick_params(labelsize=13.5)
    regularized_colorbar = fig.colorbar(images[2], cax=regularized_colorbar_axis)
    regularized_colorbar.set_label(r"log10(relative regularized curvature)", fontsize=15.5)
    if spec["regularized_vmin"] <= -3.0:
        regularized_colorbar.set_ticks(np.arange(-6, 1, 1))
    else:
        regularized_colorbar.set_ticks(np.arange(-1.0, 0.1, 0.2))
    regularized_colorbar.ax.tick_params(labelsize=13.5)
    fig.savefig(spec["output"], dpi=600, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    return spec["output"], j_indices


results = {}
sensitivity_statistics = {}
for case_name, spec in CASES.items():
    result = compute_joint_sensitivity(case_name, spec)
    output, j_indices = plot_sensitivity(case_name, spec, result)
    results[case_name] = result
    sensitivity_statistics[case_name] = sensitivity_summary(case_name, spec, result)
    print(
        f"{case_name}: wrote {output.relative_to(ROOT)} | "
        f"active cells={result['n_active']} | "
        f"sensitivity calculation={result['elapsed_s']:.1f} s"
    )
    print("  XZ northings (m):", [round(float(result['core_mesh'].cell_centers_y[j]), 1) for j in j_indices])
    summary = sensitivity_statistics[case_name]
    print("  depth thresholds (km below local topography):")
    for property_name, thresholds in summary["thresholds_km"].items():
        threshold_text = ", ".join(f"10^{int(np.log10(level))}: {depth:.2f}" for level, depth in thresholds.items())
        print(f"    {property_name}: {threshold_text}")
    for label, interval in summary["intervals"].items():
        gravity_stats = interval["gravity_log10"]
        magnetic_stats = interval["magnetic_log10"]
        data_fraction = interval["data_fraction"]
        print(
            f"  {label} {interval['depth_range_km']} km: "
            f"gravity median={gravity_stats['median']:.3f} (p10={gravity_stats['p10']:.3f}, p90={gravity_stats['p90']:.3f}); "
            f"magnetic median={magnetic_stats['median']:.3f} (p10={magnetic_stats['p10']:.3f}, p90={magnetic_stats['p90']:.3f}); "
            f"data-curvature fraction={data_fraction['median_percent']:.3e}%"
        )
